# Real-Air — Train LSTM + Prophet on Indian AQI

Trains a 72h → 48h PM2.5 LSTM (PyTorch) and a Prophet baseline for **all 10 cities**:  
Delhi · Mumbai · Hyderabad · Chennai · Bangalore · Kolkata · Pune · Ahmedabad · Jaipur · Nellore

## No Kaggle dataset needed
Everything is fetched live from two free APIs (no keys required):
- **Open-Meteo Air Quality** — PM2.5, PM10, NO₂, O₃, CO, SO₂ (CAMS reanalysis, ~3 years history, any lat/lon)
- **Open-Meteo Archive** — temperature, humidity, wind speed

## Settings required
- **Accelerator**: GPU T4 x1 (recommended, ~8 min). CPU works too, ~40 min.
- **Internet**: ON ← must be enabled

## Output
Checkpoints land in `/kaggle/working/checkpoints/`.  
Download → drop into `backend/models/checkpoints/` → restart FastAPI → real forecasts for all 10 cities.

In [ ]:
!pip install -q prophet tabulate

In [ ]:
import os, pickle, logging, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
import requests
from tabulate import tabulate
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}', f'| GPU: {torch.cuda.get_device_name(0)}' if DEVICE.type=='cuda' else '')

CKPT_DIR = Path('/kaggle/working/checkpoints')
CKPT_DIR.mkdir(exist_ok=True)
print(f'Checkpoints → {CKPT_DIR}')

In [ ]:
CITIES = {
    'Delhi':     {'lat': 28.6139, 'lon': 77.2090},
    'Mumbai':    {'lat': 19.0760, 'lon': 72.8777},
    'Hyderabad': {'lat': 17.3850, 'lon': 78.4867},
    'Chennai':   {'lat': 13.0827, 'lon': 80.2707},
    'Bangalore': {'lat': 12.9716, 'lon': 77.5946},
    'Kolkata':   {'lat': 22.5726, 'lon': 88.3639},
    'Pune':      {'lat': 18.5204, 'lon': 73.8567},
    'Ahmedabad': {'lat': 23.0225, 'lon': 72.5714},
    'Jaipur':    {'lat': 26.9124, 'lon': 75.7873},
    'Nellore':   {'lat': 14.4426, 'lon': 79.9865},
}

FEATURES   = ['pm25', 'pm10', 'no2', 'temperature', 'humidity', 'wind_speed']
SEQ_LEN    = 72
PRED_LEN   = 48
BATCH      = 64
EPOCHS     = 50
LR         = 1e-3
PATIENCE   = 6
HISTORY_DAYS = 3 * 365   # ~3 years of hourly data per city

def safe_name(c): return c.lower().replace(' ', '_')

## 1. Fetch data from Open-Meteo (all 10 cities)

In [ ]:
from datetime import datetime, timedelta

def fetch_air_quality(city, lat, lon, days):
    """Hourly PM2.5/PM10/NO2/O3/CO/SO2 from Open-Meteo CAMS archive."""
    end   = datetime.utcnow().date() - timedelta(days=1)
    start = end - timedelta(days=days)
    url = 'https://air-quality-api.open-meteo.com/v1/air-quality'
    r = requests.get(url, params={
        'latitude': lat, 'longitude': lon,
        'start_date': str(start), 'end_date': str(end),
        'hourly': 'pm2_5,pm10,nitrogen_dioxide,ozone,carbon_monoxide,sulphur_dioxide',
        'timezone': 'UTC', 'domains': 'cams_global',
    }, timeout=120)
    r.raise_for_status()
    h = r.json()['hourly']
    return pd.DataFrame({
        'city': city,
        'timestamp': pd.to_datetime(h['time']),
        'pm25': h['pm2_5'], 'pm10': h['pm10'],
        'no2': h['nitrogen_dioxide'], 'o3': h['ozone'],
        'co': h['carbon_monoxide'], 'so2': h['sulphur_dioxide'],
    })

def fetch_weather(city, lat, lon, days):
    """Hourly temp/humidity/wind from Open-Meteo archive."""
    end   = datetime.utcnow().date() - timedelta(days=1)
    start = end - timedelta(days=days)
    url = 'https://archive-api.open-meteo.com/v1/archive'
    r = requests.get(url, params={
        'latitude': lat, 'longitude': lon,
        'start_date': str(start), 'end_date': str(end),
        'hourly': 'temperature_2m,relative_humidity_2m,wind_speed_10m',
        'timezone': 'UTC',
    }, timeout=120)
    r.raise_for_status()
    h = r.json()['hourly']
    return pd.DataFrame({
        'city': city,
        'timestamp': pd.to_datetime(h['time']),
        'temperature': h['temperature_2m'],
        'humidity': h['relative_humidity_2m'],
        'wind_speed': h['wind_speed_10m'],
    })

aq_frames, wx_frames = [], []
for city, c in CITIES.items():
    t0 = time.time()
    try:
        aq = fetch_air_quality(city, c['lat'], c['lon'], HISTORY_DAYS)
        wx = fetch_weather(city, c['lat'], c['lon'], HISTORY_DAYS)
        aq_frames.append(aq); wx_frames.append(wx)
        print(f'  {city:12s}  aq={len(aq):>6d} rows  wx={len(wx):>6d} rows  ({time.time()-t0:.1f}s)')
    except Exception as e:
        print(f'  {city:12s}  FAILED: {e}')

df_aq = pd.concat(aq_frames, ignore_index=True)
df_wx = pd.concat(wx_frames, ignore_index=True)
print(f'\nTotal AQ rows : {len(df_aq):,}')
print(f'Total WX rows : {len(df_wx):,}')

In [ ]:
# Merge pollutants + weather on (city, hour)
df_aq['timestamp'] = df_aq['timestamp'].dt.floor('h')
df_wx['timestamp'] = df_wx['timestamp'].dt.floor('h')
merged = pd.merge(df_aq, df_wx, on=['city', 'timestamp'], how='inner')

# India NAAQS PM2.5 → AQI
BP = [(0,30,0,50),(30,60,51,100),(60,90,101,200),(90,120,201,300),(120,250,301,400),(250,500,401,500)]
def pm25_to_aqi(p):
    if pd.isna(p) or p < 0: return np.nan
    for clo,chi,ilo,ihi in BP:
        if clo <= p <= chi: return ((ihi-ilo)/(chi-clo))*(p-clo)+ilo
    return 500.0
merged['aqi'] = merged['pm25'].apply(pm25_to_aqi)

print('Rows per city:')
print(merged.groupby('city').size().sort_values(ascending=False).to_string())
merged.head()

## 2. LSTM model

In [ ]:
class AQIDataset(Dataset):
    def __init__(self, data, seq=SEQ_LEN, pred=PRED_LEN):
        self.X, self.y = [], []
        for i in range(len(data) - seq - pred + 1):
            self.X.append(data[i:i+seq])
            self.y.append(data[i+seq:i+seq+pred, 0])   # PM2.5 column
        self.X = np.array(self.X, dtype=np.float32)
        self.y = np.array(self.y, dtype=np.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return torch.tensor(self.X[i]), torch.tensor(self.y[i])

class LSTMForecaster(nn.Module):
    def __init__(self, n_features=len(FEATURES), hidden=128, layers=2, dropout=0.2, pred=PRED_LEN):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, layers, batch_first=True, dropout=dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, pred)
        )
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])

def mape(y, yp):
    m = y != 0
    return np.mean(np.abs((y[m]-yp[m])/y[m]))*100 if m.any() else np.nan

def get_metrics(y, yp):
    return dict(
        MAE=round(float(mean_absolute_error(y, yp)), 3),
        RMSE=round(float(np.sqrt(mean_squared_error(y, yp))), 3),
        MAPE=round(float(mape(y, yp)), 2),
    )

## 3. Train per-city LSTM + Prophet

In [ ]:
def train_lstm(city, df_city):
    feats = df_city[FEATURES].copy().ffill().bfill().dropna()
    if len(feats) < SEQ_LEN + PRED_LEN + 50:
        raise ValueError(f'Only {len(feats)} rows after cleaning')

    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(feats.values)
    split  = int(len(scaled) * 0.8)
    train_dl = DataLoader(AQIDataset(scaled[:split]), batch_size=BATCH, shuffle=True)
    test_dl  = DataLoader(AQIDataset(scaled[split:]),  batch_size=BATCH)

    model  = LSTMForecaster().to(DEVICE)
    opt    = torch.optim.Adam(model.parameters(), lr=LR)
    sch    = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    crit   = nn.HuberLoss()

    best, ctr, best_state = float('inf'), 0, None
    for epoch in range(1, EPOCHS+1):
        model.train()
        for xb, yb in train_dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad(); loss = crit(model(xb), yb); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()

        model.eval(); vl = 0
        with torch.no_grad():
            for xb, yb in test_dl:
                vl += crit(model(xb.to(DEVICE)), yb.to(DEVICE)).item()
        vl /= max(len(test_dl), 1); sch.step(vl)

        if vl < best:
            best, ctr = vl, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            ctr += 1
            if ctr >= PATIENCE: break

    # save
    torch.save(best_state, CKPT_DIR / f'lstm_{safe_name(city)}_best.pt')
    torch.save(best_state, CKPT_DIR / f'lstm_{safe_name(city)}.pt')
    with open(CKPT_DIR / f'scaler_{safe_name(city)}.pkl', 'wb') as f:
        pickle.dump(scaler, f)

    # evaluate on test set
    model.load_state_dict(best_state); model.eval()
    preds, tgts = [], []
    with torch.no_grad():
        for xb, yb in test_dl:
            preds.append(model(xb.to(DEVICE)).cpu().numpy()); tgts.append(yb.numpy())
    preds = np.concatenate(preds); tgts = np.concatenate(tgts)
    dummy = np.zeros((preds.size, len(FEATURES)))
    dummy[:, 0] = preds.flatten()
    p_real = scaler.inverse_transform(dummy)[:, 0]
    dummy[:, 0] = tgts.flatten()
    t_real = scaler.inverse_transform(dummy)[:, 0]
    return get_metrics(t_real, p_real), (t_real, p_real)


def train_prophet(city, df_city):
    from prophet import Prophet
    s = df_city[['timestamp','pm25']].rename(columns={'timestamp':'ds','pm25':'y'}).dropna()
    s['ds'] = pd.to_datetime(s['ds']).dt.tz_localize(None)
    if len(s) < 200: raise ValueError('Too little data')
    split = int(len(s) * 0.8)
    tr, te = s.iloc[:split], s.iloc[split:]
    m = Prophet(seasonality_mode='multiplicative', daily_seasonality=True,
                weekly_seasonality=True, yearly_seasonality=True,
                changepoint_prior_scale=0.05)
    m.fit(tr)
    with open(CKPT_DIR / f'prophet_{safe_name(city)}.pkl', 'wb') as f:
        pickle.dump(m, f)
    fut = m.make_future_dataframe(periods=len(te), freq='h')
    fc  = m.predict(fut)
    yp  = np.clip(fc['yhat'].iloc[-len(te):].values, 0, None)
    return get_metrics(te['y'].values, yp)

In [ ]:
results, predictions = [], {}

for city in CITIES:
    sub = merged[merged['city'] == city].sort_values('timestamp').reset_index(drop=True)
    if len(sub) < SEQ_LEN + PRED_LEN + 100:
        print(f'\n--- Skipping {city}: only {len(sub)} rows ---'); continue
    print(f'\n=== {city} ({len(sub):,} rows) ===')
    row = {'city': city}

    t0 = time.time()
    try:
        lm, (yt, yp) = train_lstm(city, sub)
        row.update({'LSTM_MAE': lm['MAE'], 'LSTM_RMSE': lm['RMSE'], 'LSTM_MAPE': f"{lm['MAPE']}%"})
        predictions[city] = (yt[:PRED_LEN], yp[:PRED_LEN])
        print(f'  LSTM    {lm}  ({time.time()-t0:.0f}s)')
    except Exception as e:
        print(f'  LSTM FAILED: {e}')
        row.update({'LSTM_MAE':'—','LSTM_RMSE':'—','LSTM_MAPE':'—'})

    t0 = time.time()
    try:
        pm = train_prophet(city, sub)
        row.update({'Prophet_MAE': pm['MAE'], 'Prophet_RMSE': pm['RMSE'], 'Prophet_MAPE': f"{pm['MAPE']}%"})
        print(f'  Prophet {pm}  ({time.time()-t0:.0f}s)')
    except Exception as e:
        print(f'  Prophet FAILED: {e}')
        row.update({'Prophet_MAE':'—','Prophet_RMSE':'—','Prophet_MAPE':'—'})

    results.append(row)

print('\n\n=== RESULTS ===')
print(tabulate(results, headers='keys', tablefmt='rounded_outline'))

## 4. Forecast visualisation

In [ ]:
if predictions:
    n = min(len(predictions), 4)
    fig, axes = plt.subplots(n, 1, figsize=(12, 3*n), squeeze=False)
    for ax, (city, (yt, yp)) in zip(axes[:,0], list(predictions.items())[:n]):
        ax.plot(yt, label='Actual', linewidth=2)
        ax.plot(yp, label='LSTM', linestyle='--', linewidth=2)
        ax.set_title(f'{city} — first 48h test forecast')
        ax.set_ylabel('PM2.5 µg/m³'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()

## 5. Save & download

1. After this cell runs → **Output** panel (right sidebar) → expand `/kaggle/working` → download `checkpoints/` as zip  
2. Unzip into `Real_air/backend/models/checkpoints/`  
3. Restart the FastAPI server — `/api/cities/<city>/forecast` will serve real LSTM predictions

In [ ]:
pd.DataFrame(results).to_csv('/kaggle/working/metrics.csv', index=False)
print('Saved checkpoints:')
for p in sorted(CKPT_DIR.iterdir()):
    print(f'  {p.name:42s}  {p.stat().st_size/1024:>8.1f} KB')
print(f'\nCities trained: {len(results)} / {len(CITIES)}')